# 01_Preprocesamiento_PCAP

Configuración, conversión PCAP→CSV y validación del CSV.

In [ ]:
from pathlib import Path
import shutil, subprocess, sys, time
import importlib.metadata as md
import pandas as pd

pd.set_option("display.max_columns",200)
pd.set_option("display.width",200)

REPO_ROOT=Path.cwd()
PCAP_DIR=REPO_ROOT/"pcap_files"
CSV_DIR=REPO_ROOT/"csv_raw"
TMP_DIR=REPO_ROOT/".tmp_cicflowmeter"
for d in (PCAP_DIR,CSV_DIR,TMP_DIR):
    d.mkdir(parents=True,exist_ok=True)

print("Python:",sys.version.split()[0])
for p in ["pandas","scapy","cicflowmeter"]:
    try:
        print(f"{p}: {md.version(p)}")
    except Exception:
        print(f"{p}: no disponible")
print("cicflowmeter:",shutil.which("cicflowmeter"))


In [ ]:
pcaps=sorted(PCAP_DIR.glob("*.pcap"))
if not pcaps:
    raise FileNotFoundError(f"No hay archivos .pcap en {PCAP_DIR}")
PCAP_PATH=pcaps[0]
TMP_CSV=TMP_DIR/f"{PCAP_PATH.stem}.csv"
CSV_PATH=CSV_DIR/f"{PCAP_PATH.stem}.csv"
print("PCAP:",PCAP_PATH)


In [ ]:
def run_cicflowmeter(pcap_path,out_csv):
    if out_csv.exists():
        out_csv.unlink()
    cmd=["cicflowmeter","-f",str(pcap_path),"-c",str(out_csv)]
    print(" ".join(cmd))
    t0=time.time()
    r=subprocess.run(cmd,capture_output=True,text=True)
    print("Return:",r.returncode)
    print("Tiempo(s):",round(time.time()-t0,2))
    print("STDERR:\n",r.stderr or "(vacío)")
    return r

r=run_cicflowmeter(PCAP_PATH,TMP_CSV)
if r.returncode!=0:
    raise RuntimeError("cicflowmeter falló")


In [ ]:
from pandas.errors import EmptyDataError

if not TMP_CSV.exists():
    raise FileNotFoundError(TMP_CSV)

with open(TMP_CSV,"rb") as f:
    head=f.read(300)

if head.strip()==b"":
    raise RuntimeError(f"CSV inválido. Primeros bytes: {head!r}")

try:
    df=pd.read_csv(TMP_CSV,low_memory=False)
except EmptyDataError:
    raise RuntimeError(f"CSV sin columnas. Primeros bytes: {head!r}")

if df.empty or df.shape[1]==0:
    raise RuntimeError("CSV vacío o sin columnas")

shutil.copy2(TMP_CSV,CSV_PATH)

print("CSV válido:",CSV_PATH)
print("Shape:",df.shape)
display(df.head())
